# CPG Price Monitoring with Nimble + Snowflake Cortex

**Scenario.** You manage a CPG product catalog inside Snowflake and want a daily, automated read of how every SKU appears across Amazon, Walmart, and Target — current price, stock status, ratings, review counts. Today this is a manual analyst job; this notebook turns it into ~50 lines of SQL + Python that you can put on a cron.

**What you'll build.**

1. A small `PRODUCTS` master table
2. A `PRODUCT_LISTINGS` append-only history of cross-retailer enrichments
3. An `ENRICH_PRODUCT_LISTINGS` stored proc that, for each `(sku, retailer)` pair, calls `NIMBLE_SEARCH` to find the PDP, `NIMBLE_EXTRACT` to grab the page, and Snowflake Cortex (`COMPLETE`) to pull structured fields out of the markdown
4. A `v_price_alerts` view that flags 10%+ price drops or out-of-stock events
5. A `schedule.sql` that puts the whole thing on a daily cron (separate file)

**Prereqs.** Run `../../01_setup.sql` through `../../04_cortex_agent.sql` first — they create the role, database, warehouse, secret, external access integration, and the two Nimble wrapper procs this notebook calls.

**Runtime target.** Plain Jupyter `.ipynb` (nbformat v4). Opens in either Snowflake Notebooks runtime — [Notebooks in Workspaces](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-overview) (GA Feb 2026, Jupyter-compatible) or Legacy Snowflake Notebooks — and in local JupyterLab against a Snowpark session. See this recipe's `README.md` for import steps.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_role("NIMBLE_ROLE")
session.use_warehouse("NIMBLE_AGENT_WH")
session.use_database("NIMBLE_INTEGRATION")
session.use_schema("RECIPES")

session.sql("""
CREATE OR REPLACE TABLE PRODUCTS (
    sku           STRING PRIMARY KEY,
    brand         STRING,
    product_name  STRING,
    upc           STRING,
    category      STRING
)
""").collect()

# Sample products are fictional placeholders; replace with your own catalog.
session.sql("""
INSERT INTO PRODUCTS (sku, brand, product_name, upc, category) VALUES
    ('NB-001', 'Northwave Beverages', 'Sparkling Water 16.9oz 12pk', 'EXAMPLE-001', 'Beverages'),
    ('CR-014', 'Cresco',              'Vintage Cola 12oz 12pk',     'EXAMPLE-002', 'Beverages'),
    ('BB-002', 'Beacon Brewing',      'Hazy IPA 12pk',              'EXAMPLE-003', 'Beverages')
""").collect()

session.sql("""
CREATE OR REPLACE TABLE PRODUCT_LISTINGS (
    sku           STRING,
    retailer      STRING,
    url           STRING,
    price         NUMBER(10, 2),
    currency      STRING,
    in_stock      BOOLEAN,
    rating        NUMBER(3, 2),
    review_count  INTEGER,
    seller        STRING,
    image_url     STRING,
    enriched_at   TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
)
""").collect()

print("Seeded PRODUCTS with 3 rows; PRODUCT_LISTINGS ready.")

## The Search → Extract → Cortex pipeline

For each `(sku, retailer)` pair we do three things:

1. **`NIMBLE_SEARCH`** — query `"{brand} {product_name}"` with `focus="shopping"` and `include_domains=[retailer]`. Returns ranked PDP candidates.
2. **`NIMBLE_EXTRACT`** — render the top PDP candidate with JS turned on and return its markdown. Render is needed because Amazon/Walmart/Target are SPAs that hydrate price client-side.
3. **`SNOWFLAKE.CORTEX.COMPLETE`** — feed the markdown into an LLM with a structured-output prompt to pull `price`, `in_stock`, `rating`, `review_count` out of the page text.

Parallelism matters because 3 SKUs × 3 retailers = 9 round-trips, and each Extract can be 5–15 seconds with rendering on. `ThreadPoolExecutor(max_workers=10)` collapses that from ~90s serial to ~15s wall-clock.

> **Production note.** Nimble also ships retailer-specific Web Search Agents (Amazon PDP, Walmart PDP, Target PDP) that return clean structured JSON without the Extract → Cortex step. They live at `POST /v1/agent/run` — a separate API surface documented in the [agent gallery](https://docs.nimbleway.com/nimble-sdk/agentic/agent-gallery/). For production CPG workflows, swap the proc body to call those agents instead of the markdown+LLM pipeline below.

In [ ]:
session.sql("""
CREATE OR REPLACE PROCEDURE NIMBLE_INTEGRATION.RECIPES.ENRICH_PRODUCT_LISTINGS(
    retailers ARRAY,
    max_workers INTEGER DEFAULT 10
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'main'
AS
$$
import json, re
from concurrent.futures import ThreadPoolExecutor, as_completed

DOMAINS = {
    'amazon':  'amazon.com',
    'walmart': 'walmart.com',
    'target':  'target.com',
}
PDP_PATTERNS = {
    'amazon':  re.compile(r'amazon\\.com/.*?/dp/[A-Z0-9]{10}'),
    'walmart': re.compile(r'walmart\\.com/ip/[^/?#]+'),
    'target':  re.compile(r'target\\.com/p/[^/?#]+/-/A-\\d+'),
}
EXTRACT_PROMPT = (
    'You will receive the markdown contents of a retailer product page. '
    'Return a single JSON object with these fields (use null when unknown): '
    'price (number), currency (string, e.g. USD), in_stock (boolean), '
    'rating (number 0-5), review_count (integer), seller (string), image_url (string). '
    'Respond with JSON only, no commentary.\\n\\nMarkdown:\\n'
)

def _find_pdp_url(search_json, retailer):
    pattern = PDP_PATTERNS[retailer]
    for r in search_json.get('results', []):
        url = r.get('url', '')
        if pattern.search(url):
            return url
    return None

def _parse_cortex_json(raw):
    raw = raw.strip()
    if raw.startswith('```'):
        raw = raw.strip('`').lstrip('json').strip()
    try:
        return json.loads(raw)
    except Exception:
        m = re.search(r'\\{.*\\}', raw, re.DOTALL)
        return json.loads(m.group(0)) if m else {}

def _enrich_one(session, sku, brand, name, retailer):
    domain = DOMAINS.get(retailer)
    if not domain:
        return (sku, retailer, None, 'unknown retailer')
    query = f'{brand} {name}'
    search_raw = session.call(
        'NIMBLE_INTEGRATION.TOOLS.NIMBLE_SEARCH',
        query, 5, 'shopping', 'fast', False, 'US', 'en', [domain], None, None,
    )
    search = json.loads(search_raw)
    pdp_url = _find_pdp_url(search, retailer)
    if not pdp_url:
        return (sku, retailer, None, 'no PDP found')

    extract_raw = session.call(
        'NIMBLE_INTEGRATION.TOOLS.NIMBLE_EXTRACT',
        pdp_url, True, False, None, ['markdown'], 'US', None, 'vx6',
    )
    extract = json.loads(extract_raw)
    markdown = (extract.get('data') or {}).get('markdown', '')[:12000]
    if not markdown:
        return (sku, retailer, pdp_url, 'no markdown')

    cortex_raw = session.sql(
        "SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', :1) AS r",
        params=[EXTRACT_PROMPT + markdown],
    ).collect()[0]['R']
    fields = _parse_cortex_json(cortex_raw)

    session.sql(
        '''INSERT INTO NIMBLE_INTEGRATION.RECIPES.PRODUCT_LISTINGS
           (sku, retailer, url, price, currency, in_stock, rating, review_count, seller, image_url)
           VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9, :10)''',
        params=[
            sku, retailer, pdp_url,
            fields.get('price'), fields.get('currency'),
            fields.get('in_stock'), fields.get('rating'),
            fields.get('review_count'), fields.get('seller'),
            fields.get('image_url'),
        ],
    ).collect()
    return (sku, retailer, pdp_url, 'ok')

def main(session, retailers, max_workers):
    products = session.sql(
        'SELECT sku, brand, product_name FROM NIMBLE_INTEGRATION.RECIPES.PRODUCTS'
    ).collect()
    pairs = [(p['SKU'], p['BRAND'], p['PRODUCT_NAME'], r)
             for p in products for r in retailers]
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(_enrich_one, session, sku, b, n, ret)
                   for (sku, b, n, ret) in pairs]
        for f in as_completed(futures):
            results.append(f.result())
    return json.dumps({'enriched': len(results), 'detail': results})
$$
""").collect()

print("ENRICH_PRODUCT_LISTINGS proc created.")

In [ ]:
result = session.sql(
    "CALL NIMBLE_INTEGRATION.RECIPES.ENRICH_PRODUCT_LISTINGS(ARRAY_CONSTRUCT('amazon', 'walmart', 'target'), 10)"
).collect()
print(result[0][0])

In [6]:
session.sql("""
SELECT sku, retailer, url, price, currency, in_stock, rating, review_count, seller, enriched_at
FROM NIMBLE_INTEGRATION.RECIPES.PRODUCT_LISTINGS
ORDER BY sku, retailer
""").show()

SAMPLE OUTPUT (one row per (sku, retailer)). All values are illustrative placeholders.

+----------+----------+--------------------------------------------------------+--------+----------+----------+--------+--------------+-----------+----------------------+
| SKU      | RETAILER | URL                                                    | PRICE  | CURRENCY | IN_STOCK | RATING | REVIEW_COUNT | SELLER    | ENRICHED_AT          |
+----------+----------+--------------------------------------------------------+--------+----------+----------+--------+--------------+-----------+----------------------+
| NB-001   | amazon   | https://amazon.com/Northwave-Sparkling/dp/B0EXAMPLE01  |  19.99 | USD      | TRUE     |  4.70  |        18432 | Northwave | 2026-05-26 08:00:13  |
| NB-001   | walmart  | https://walmart.com/ip/Northwave-Sparkling-Water-12pk/ |  17.48 | USD      | TRUE     |  4.60  |         2104 | Walmart   | 2026-05-26 08:00:21  |
| NB-001   | target   | https://target.com/p/northwave-sp

## From raw rows to actionable alerts

Daily snapshots become valuable when they trigger action: a competitor undercut us, our SKU went out of stock at a key retailer, ratings dropped. The view below flags two conditions per `(sku, retailer)`:

- **Price drop** — today's price is more than 10% below the 7-day median for the same `(sku, retailer)` pair
- **OOS event** — `in_stock = FALSE` today but `TRUE` on the prior snapshot

Window functions over `PRODUCT_LISTINGS` (append-only history) make this two CTEs.

In [ ]:
session.sql("""
CREATE OR REPLACE VIEW NIMBLE_INTEGRATION.RECIPES.V_PRICE_ALERTS AS
WITH recent AS (
    SELECT
        sku,
        retailer,
        enriched_at,
        price,
        in_stock,
        MEDIAN(price) OVER (
            PARTITION BY sku, retailer
            ORDER BY enriched_at
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS price_7d_median,
        LAG(in_stock) OVER (PARTITION BY sku, retailer ORDER BY enriched_at) AS prev_in_stock,
        ROW_NUMBER() OVER (PARTITION BY sku, retailer ORDER BY enriched_at DESC) AS rn
    FROM NIMBLE_INTEGRATION.RECIPES.PRODUCT_LISTINGS
)
SELECT
    sku,
    retailer,
    enriched_at,
    price,
    price_7d_median,
    ROUND(100.0 * (price - price_7d_median) / NULLIF(price_7d_median, 0), 1) AS pct_change,
    in_stock,
    CASE
        WHEN price < 0.9 * price_7d_median THEN 'PRICE_DROP'
        WHEN in_stock = FALSE AND prev_in_stock = TRUE THEN 'OOS_EVENT'
    END AS alert_type
FROM recent
WHERE rn = 1
  AND (price < 0.9 * price_7d_median OR (in_stock = FALSE AND prev_in_stock = TRUE))
""").collect()

print("V_PRICE_ALERTS view created.")

In [9]:
session.sql("""
SELECT sku, retailer, enriched_at, price, price_7d_median, pct_change, in_stock, alert_type
FROM NIMBLE_INTEGRATION.RECIPES.V_PRICE_ALERTS
ORDER BY enriched_at DESC
""").show()

SAMPLE OUTPUT after ~10 days of daily runs. All values are illustrative placeholders.

+---------+----------+---------------------+--------+------------------+------------+----------+-------------+
| SKU     | RETAILER | ENRICHED_AT         | PRICE  | PRICE_7D_MEDIAN  | PCT_CHANGE | IN_STOCK | ALERT_TYPE  |
+---------+----------+---------------------+--------+------------------+------------+----------+-------------+
| NB-001  | walmart  | 2026-05-26 08:00:21 |  14.97 |            17.48 |     -14.4  | TRUE     | PRICE_DROP  |
| CR-014  | walmart  | 2026-05-26 08:00:24 |  27.46 |            27.46 |       0.0  | FALSE    | OOS_EVENT   |
+---------+----------+---------------------+--------+------------------+------------+----------+-------------+
2 Row(s) produced.

## Now what

**Put it on a schedule.** [`schedule.sql`](./schedule.sql) wraps `ENRICH_PRODUCT_LISTINGS` in a Snowflake Task that runs daily at 08:00 Pacific. Deploy it once, and your CPG team has a self-refreshing competitive-pricing feed feeding into `V_PRICE_ALERTS`.

**Wire alerts into your downstream tools.** Pipe `V_PRICE_ALERTS` into a Streamlit-in-Snowflake dashboard, a Slack webhook via a Snowflake External Function, or a row-change trigger on the underlying table.

**Ask natural-language questions over this data.** A companion recipe extends this pattern with a Cortex Analyst [Semantic View](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/semantic-model-spec) so analysts can ask things like "which SKUs dropped >5% at Walmart this week" without writing SQL.

**Swap in retailer-specific Web Search Agents.** Replace the Extract + Cortex step with Nimble's pre-built Amazon/Walmart/Target PDP agents (`POST /v1/agent/run`, [agent gallery](https://docs.nimbleway.com/nimble-sdk/agentic/agent-gallery/)) for higher-fidelity structured output and fewer LLM tokens.